In [ ]:
import os
import glob
import cv2
import mediapipe as mp
from concurrent.futures import ThreadPoolExecutor
import math

def profil_dataset(input_folder):
    daftar_video = glob.glob(os.path.join(input_folder, "*.mp4"))
    stats = []
    
    for path in daftar_video:
        cap = cv2.VideoCapture(path)
        fps = int(cap.get(cv2.CAP_PROP_FPS) or 30)
        total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        
        durasi_detik = total_frame / fps
        if durasi_detik <= 165:
            continue
            
        frame_valid = total_frame - (45 * fps) - (120 * fps)
        stats.append({
            "path": path,
            "fps": fps,
            "frame_valid": frame_valid,
            "nama": os.path.splitext(os.path.basename(path))[0]
        })
    
    return stats

def hitung_kuota(stats, target_menit=18):
    target_total_detik = len(stats) * target_menit * 60
    total_frame_tersedia = sum(s["frame_valid"] for s in stats)
    
    for s in stats:
        if total_frame_tersedia == 0:
            s["kuota"] = 0
            continue
            
        rasio = s["frame_valid"] / total_frame_tersedia
        target_frame_video = int(target_total_detik * s["fps"] * rasio)
        s["kuota"] = min(target_frame_video, s["frame_valid"])
        
    return stats

def cek_wajah_pojok(bbox):
    cx = bbox.xmin + (bbox.width / 2)
    cy = bbox.ymin + (bbox.height / 2)
    
    if 0.25 < cx < 0.75 and 0.25 < cy < 0.75:
        return False
    return True

def proses_video(data):
    path = data["path"]
    fps = data["fps"]
    kuota = data["kuota"]
    nama = data["nama"]
    folder_out = data["folder_out"]
    
    os.makedirs(folder_out, exist_ok=True)
    
    cap = cv2.VideoCapture(path)
    mp_face = mp.solutions.face_detection
    detector = mp_face.FaceDetection(min_detection_confidence=0.5)
    
    start_frame = 45 * fps
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    jumlah_simpan = 0
    nomor_frame = start_frame
    
    while cap.isOpened() and jumlah_simpan < kuota:
        ret, frame = cap.read()
        if not ret:
            break
            
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hasil = detector.process(rgb)
        
        if hasil.detections:
            tinggi, lebar, _ = frame.shape
            
            for deteksi in hasil.detections:
                bbox = deteksi.location_data.relative_bounding_box
                
                if cek_wajah_pojok(bbox):
                    x = max(0, int(bbox.xmin * lebar))
                    y = max(0, int(bbox.ymin * tinggi))
                    w = min(lebar - x, int(bbox.width * lebar))
                    h = min(tinggi - y, int(bbox.height * tinggi))
                    
                    wajah = frame[y:y+h, x:x+w]
                    
                    if wajah.size > 0:
                        wajah_resize = cv2.resize(wajah, (224, 224))
                        file_out = os.path.join(folder_out, f"frame_{jumlah_simpan:05d}.jpg")
                        cv2.imwrite(file_out, wajah_resize)
                        jumlah_simpan += 1
                        break 
                        
        nomor_frame += 1

    cap.release()
    detector.close()
    return nama, jumlah_simpan

def jalankan_ekstraksi(input_folder="input_folder", output_folder="dataset_framev2"):
    stats = profil_dataset(input_folder)
    if not stats:
        print("Dataset kosong atau video terlalu pendek.")
        return
        
    stats_kuota = hitung_kuota(stats)
    
    for s in stats_kuota:
        s["folder_out"] = os.path.join(output_folder, s["nama"])
        
    print(f"Memproses {len(stats_kuota)} video...")
    
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        hasil = list(executor.map(proses_video, stats_kuota))
        
    total_wajah = 0
    for nama, jumlah in hasil:
        total_wajah += jumlah
        
    print(f"Selesai. Total wajah terekstrak: {total_wajah}")

if __name__ == "__main__":
    jalankan_ekstraksi()

Memproses 2 video...


In [ ]:
import os
import glob
import cv2
import mediapipe as mp
from concurrent.futures import ThreadPoolExecutor

def profil_dataset(input_folder):
    daftar_video = glob.glob(os.path.join(input_folder, "*.mp4"))
    stats = []
    
    for path in daftar_video:
        cap = cv2.VideoCapture(path)
        fps = int(cap.get(cv2.CAP_PROP_FPS) or 30)
        total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        
        durasi_detik = total_frame / fps
        if durasi_detik <= 165:
            continue
            
        frame_valid = total_frame - (45 * fps) - (120 * fps)
        stats.append({
            "path": path,
            "fps": fps,
            "frame_valid": frame_valid,
            "nama": os.path.splitext(os.path.basename(path))[0]
        })
    
    return stats

def hitung_kuota(stats, target_menit=18):
    target_total_detik = len(stats) * target_menit * 60
    total_frame_tersedia = sum(s["frame_valid"] for s in stats)
    
    for s in stats:
        if total_frame_tersedia == 0:
            s["kuota"] = 0
            continue
            
        rasio = s["frame_valid"] / total_frame_tersedia
        target_frame_video = int(target_total_detik * s["fps"] * rasio)
        s["kuota"] = min(target_frame_video, s["frame_valid"])
        
    return stats

def cek_wajah_valid(bbox):
    cx = bbox.xmin + (bbox.width / 2)
    cy = bbox.ymin + (bbox.height / 2)
    
    # 1. Filter Posisi: Abaikan jika wajah ada di area tengah layar (25% - 75%)
    if 0.25 < cx < 0.75 and 0.25 < cy < 0.75:
        return False
        
    # 2. Filter Ukuran: Wajah facecam umumnya memakan 5% - 35% lebar layar
    # Jika terlalu kecil (noise) atau terlalu besar, abaikan.
    if bbox.width < 0.05 or bbox.width > 0.35:
        return False
        
    return True

def proses_video(data):
    path = data["path"]
    fps = data["fps"]
    kuota = data["kuota"]
    nama = data["nama"]
    folder_out = data["folder_out"]
    
    os.makedirs(folder_out, exist_ok=True)
    
    cap = cv2.VideoCapture(path)
    mp_face = mp.solutions.face_detection
    
    # NAIKKAN CONFIDENCE KE 0.85 agar tidak mendeteksi tekstur game secara acak
    detector = mp_face.FaceDetection(min_detection_confidence=0.85) 
    
    start_frame = 45 * fps
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    jumlah_simpan = 0
    nomor_frame = start_frame
    
    while cap.isOpened() and jumlah_simpan < kuota:
        ret, frame = cap.read()
        if not ret:
            break
            
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hasil = detector.process(rgb)
        
        if hasil.detections:
            tinggi, lebar, _ = frame.shape
            
            for deteksi in hasil.detections:
                bbox = deteksi.location_data.relative_bounding_box
                
                # Gunakan filter posisi dan ukuran yang baru
                if cek_wajah_valid(bbox):
                    x = max(0, int(bbox.xmin * lebar))
                    y = max(0, int(bbox.ymin * tinggi))
                    w = min(lebar - x, int(bbox.width * lebar))
                    h = min(tinggi - y, int(bbox.height * tinggi))
                    
                    wajah = frame[y:y+h, x:x+w]
                    
                    if wajah.size > 0:
                        wajah_resize = cv2.resize(wajah, (224, 224))
                        file_out = os.path.join(folder_out, f"frame_{jumlah_simpan:05d}.jpg")
                        cv2.imwrite(file_out, wajah_resize)
                        jumlah_simpan += 1
                        break 
                        
        nomor_frame += 1

    cap.release()
    detector.close()
    return nama, jumlah_simpan

def jalankan_ekstraksi(input_folder="input_folder", output_folder="dataset_framev2"):
    stats = profil_dataset(input_folder)
    if not stats:
        print("Dataset kosong atau video terlalu pendek.")
        return
        
    stats_kuota = hitung_kuota(stats)
    
    for s in stats_kuota:
        s["folder_out"] = os.path.join(output_folder, s["nama"])
        
    print(f"Memproses {len(stats_kuota)} video...")
    
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        hasil = list(executor.map(proses_video, stats_kuota))
        
    total_wajah = 0
    for nama, jumlah in hasil:
        total_wajah += jumlah
        
    print(f"Selesai. Total wajah terekstrak: {total_wajah}")

if __name__ == "__main__":
    jalankan_ekstraksi()

Memproses 5 video...
